# Eksperimen Analisis Sentimen #IndonesiaSadar

Notebook ini berisi alur lengkap eksperimen analisis sentimen pada data komentar media sosial terkait gerakan **#IndonesiaSadar**:
1. **Data Preprocessing & Filtering** (penanganan missing value, filtering label tidak relevan)
2. **Train-Test Split & TF-IDF Extraction** (mencegah data leakage)
3. **Pelatihan Model** (Support Vector Machine / LinearSVC & Naive Bayes / MultinomialNB)
4. **Evaluasi & Visualisasi Confusion Matrix**
5. **Export Artefak Model (.joblib)**
6. **Inferensi Data Ujian Baru Tanpa Label**

--- 
## 1. Import Library & Konfigurasi

In [ ]:
import os
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')

# Pastikan working directory berada di root project
if Path.cwd().name == 'notebooks':
    os.chdir('..')
print(f"[INFO] Root Project Directory: {Path.cwd()}")

import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    f1_score
)

--- 
## 2. Data Preprocessing & Filtering

In [ ]:
def load_and_preprocess_labeled_data(filepath: str) -> pd.DataFrame:
    """
    Memuat data berlabel (CSV/Excel), membersihkan nilai kosong (NaN) pada
    kolom komentar_clean, dan memfilter keluar label 'Tidak Relevan'.
    """
    if filepath.endswith('.xlsx') or filepath.endswith('.xls'):
        df = pd.read_excel(filepath)
    else:
        df = pd.read_csv(filepath)
        
    print(f"[INFO] Jumlah data awal: {len(df)} baris")
    
    # 1. Hapus baris dengan nilai NaN pada kolom 'komentar_clean' atau 'label'
    df = df.dropna(subset=['komentar_clean', 'label']).copy()
    
    # 2. Pastikan tipe string dan buang baris yang hanya berisi spasi kosong
    df['komentar_clean'] = df['komentar_clean'].astype(str).str.strip()
    df['label'] = df['label'].astype(str).str.strip()
    df = df[df['komentar_clean'] != '']
    
    # 3. Filter dan buang seluruh variasi label 'tidak relevan' (case-insensitive)
    irrelevant_labels = ['tidak relevan', 'tidak_relevan', 'irrelevant', 'tdk relevan']
    mask_irrelevant = df['label'].str.lower().isin(irrelevant_labels)
    df_clean = df[~mask_irrelevant].copy().reset_index(drop=True)
    
    print(f"[INFO] Jumlah data setelah hapus 'Tidak Relevan' & NaN: {len(df_clean)} baris")
    print("\n[INFO] Distribusi Label Bersih:")
    print(df_clean['label'].value_counts())
    
    return df_clean

TRAIN_FILE = 'data/raw/#IndonesiaSadar👌 - Data Training.csv'
df_clean = load_and_preprocess_labeled_data(TRAIN_FILE)
df_clean.head()

--- 
## 3. Pembagian Data (Train-Test Split 80:20) & Ekstraksi Fitur (TF-IDF)

> **Penting**: Train-test split dilakukan *sebelum* `fit` TfidfVectorizer agar data uji (20%) sama sekali tidak membocorkan informasi kosakata/IDF (mencegah *data leakage*).

In [ ]:
X = df_clean['komentar_clean']
y = df_clean['label']
comment_ids = df_clean['comment_id']

X_train, X_test, y_train, y_test, id_train, id_test = train_test_split(
    X, y, comment_ids,
    test_size=0.20,
    stratify=y,
    random_state=42
)

print(f"[INFO] Ukuran Data Latih (80%): {len(X_train)} baris")
print(f"[INFO] Ukuran Data Uji (20%)  : {len(X_test)} baris")

# Inisialisasi dan fit TfidfVectorizer HANYA pada X_train
tfidf_vectorizer = TfidfVectorizer()
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)

# Transform X_test menggunakan kamus TF-IDF dari X_train
X_test_tfidf = tfidf_vectorizer.transform(X_test)
print(f"[INFO] Ukuran Kosakata TF-IDF : {len(tfidf_vectorizer.vocabulary_)} token unik")

--- 
## 4. Pelatihan Model (LinearSVC & MultinomialNB)

In [ ]:
# a. Support Vector Machine (LinearSVC)
svm_model = LinearSVC(random_state=42, dual='auto')
svm_model.fit(X_train_tfidf, y_train)
y_pred_svm = svm_model.predict(X_test_tfidf)

# b. Naive Bayes (MultinomialNB)
nb_model = MultinomialNB()
nb_model.fit(X_train_tfidf, y_train)
y_pred_nb = nb_model.predict(X_test_tfidf)

print("[INFO] Pelatihan LinearSVC dan MultinomialNB selesai.")

--- 
## 5. Evaluasi & Perbandingan Model (pada 20% Data Uji)

In [ ]:
acc_svm = accuracy_score(y_test, y_pred_svm)
f1_svm = f1_score(y_test, y_pred_svm, average='weighted')

acc_nb = accuracy_score(y_test, y_pred_nb)
f1_nb = f1_score(y_test, y_pred_nb, average='weighted')

print("=" * 60)
print("LAPORAN KLASIFIKASI: SUPPORT VECTOR MACHINE (LinearSVC)")
print("=" * 60)
print(f"Accuracy: {acc_svm:.4f} | Weighted F1-Score: {f1_svm:.4f}\n")
print(classification_report(y_test, y_pred_svm, zero_division=0))

print("=" * 60)
print("LAPORAN KLASIFIKASI: NAIVE BAYES (MultinomialNB)")
print("=" * 60)
print(f"Accuracy: {acc_nb:.4f} | Weighted F1-Score: {f1_nb:.4f}\n")
print(classification_report(y_test, y_pred_nb, zero_division=0))

# Visualisasi Confusion Matrix berdampingan
labels_order = sorted(y.unique())
cm_svm = confusion_matrix(y_test, y_pred_svm, labels=labels_order)
cm_nb = confusion_matrix(y_test, y_pred_nb, labels=labels_order)

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5), facecolor='#0A1722')
for ax, cm, title in zip(
    axes,
    [cm_svm, cm_nb],
    [f'SVM (LinearSVC)\nAccuracy: {acc_svm:.2%} | F1-Weighted: {f1_svm:.2%}',
     f'Naive Bayes (MultinomialNB)\nAccuracy: {acc_nb:.2%} | F1-Weighted: {f1_nb:.2%}']
):
    ax.set_facecolor('#0A1722')
    sns.heatmap(
        cm, annot=True, fmt='d', cmap='YlGnBu', cbar=False,
        xticklabels=labels_order, yticklabels=labels_order,
        ax=ax, annot_kws={'size': 12, 'weight': 'bold'}
    )
    ax.set_title(title, color='#09D9F3', fontsize=12, fontweight='bold', pad=12)
    ax.set_xlabel('Prediksi Model', color='white', fontsize=11)
    ax.set_ylabel('Label Asli (Aktual)', color='white', fontsize=11)
    ax.tick_params(colors='white', labelsize=10)

plt.suptitle('Perbandingan Confusion Matrix pada 20% Data Uji', color='white', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
os.makedirs('outputs/figures', exist_ok=True)
plt.savefig('outputs/figures/confusion_matrix_comparison.png', dpi=300, bbox_inches='tight', facecolor='#0A1722')
plt.show()

# Penentuan Model Terbaik secara otomatis berdasarkan Weighted F1-Score
if f1_svm >= f1_nb:
    best_model = svm_model
    best_model_name = "SVM (LinearSVC)"
else:
    best_model = nb_model
    best_model_name = "Naive Bayes (MultinomialNB)"

print(f"\n[KESIMPULAN] Model Terbaik & Paling Reliable: {best_model_name} (F1-Weighted: {max(f1_svm, f1_nb):.4f})")

--- 
## 6. Export Artefak Model & Hasil Prediksi 20% Data Uji

In [ ]:
os.makedirs('outputs/predictions', exist_ok=True)
os.makedirs('models', exist_ok=True)

# a. Simpan hasil prediksi 20% data uji ke CSV baru
df_evaluasi_20 = pd.DataFrame({
    'comment_id': id_test.values,
    'komentar_clean': X_test.values,
    'label_asli': y_test.values,
    'prediksi_svm': y_pred_svm,
    'prediksi_naive_bayes': y_pred_nb
})
OUTPUT_TEST_CSV = 'outputs/predictions/hasil_prediksi_20persen_data_uji.csv'
df_evaluasi_20.to_csv(OUTPUT_TEST_CSV, index=False)
print(f"[EXPORT] File prediksi 20% data uji disimpan ke: {OUTPUT_TEST_CSV}")

# b. Simpan model terbaik dan TfidfVectorizer menggunakan joblib
MODEL_FILENAME = 'models/best_sentiment_model_svm.joblib'
VECTORIZER_FILENAME = 'models/tfidf_vectorizer.joblib'

joblib.dump(best_model, MODEL_FILENAME)
joblib.dump(tfidf_vectorizer, VECTORIZER_FILENAME)
print(f"[EXPORT] Model terbaik disimpan ke     : {MODEL_FILENAME}")
print(f"[EXPORT] TF-IDF Vectorizer disimpan ke : {VECTORIZER_FILENAME}")

--- 
## 7. Inferensi pada Data Ujian Tanpa Label

In [ ]:
def predict_unlabeled_data(
    input_filepath: str,
    model_filepath: str,
    vectorizer_filepath: str,
    output_filepath: str
) -> pd.DataFrame:
    """
    Memuat artefak model terbaik dan TfidfVectorizer (.joblib) untuk memprediksi
    file data baru yang belum memiliki label.
    """
    loaded_model = joblib.load(model_filepath)
    loaded_vectorizer = joblib.load(vectorizer_filepath)
    
    if input_filepath.endswith('.xlsx') or input_filepath.endswith('.xls'):
        df_unlabeled = pd.read_excel(input_filepath)
    else:
        df_unlabeled = pd.read_csv(input_filepath)
        
    clean_texts = df_unlabeled['komentar_clean'].fillna('').astype(str).str.strip()
    X_new_tfidf = loaded_vectorizer.transform(clean_texts)
    df_unlabeled['prediksi_label'] = loaded_model.predict(X_new_tfidf)
    
    df_unlabeled.to_csv(output_filepath, index=False)
    print(f"\n[INFERENSI SELESAI] {len(df_unlabeled)} baris berhasil diprediksi dan disimpan ke: {output_filepath}")
    print(df_unlabeled['prediksi_label'].value_counts())
    
    return df_unlabeled

UNLABELED_FILE = 'data/raw/#IndonesiaSadar👌 - Data Ujian.csv'
OUTPUT_UNLABELED_CSV = 'outputs/predictions/hasil_prediksi_data_ujian.csv'

df_prediksi_akhir = predict_unlabeled_data(
    input_filepath=UNLABELED_FILE,
    model_filepath=MODEL_FILENAME,
    vectorizer_filepath=VECTORIZER_FILENAME,
    output_filepath=OUTPUT_UNLABELED_CSV
)
df_prediksi_akhir.head()